# --- Model Train Orchestrator ---

In [1]:
# ===================
# Setup
# ===================
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR

PROJECT_ROOT = Path(os.getcwd()).parent

# Connect to custom-defined modules
sys.path.append(str(PROJECT_ROOT))

# Auto-reload scripts if any changes
%reload_ext autoreload
%autoreload 2


In [ ]:
# ========================
# Custom Modules
# ========================
from src.data.dataloader import create_multitask_dataloader

from src.models.config import MultiTaskModelConfig
from src.models.network import MultiTaskYOLO
from src.models.loss import MultiTaskUncertaintyLoss

from src.training.trainer import MultiTaskTrainer

from src.evaluation.evaluate import run_evaluation

from src.inference.stream import AirportStreamProcessor

In [4]:
# ======================================
# 1. Configuration & Hyperparameters
# ======================================
NUM_EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⚡ Target Execution Device: [{DEVICE.upper()}]")


⚡ Target Execution Device: [CUDA]


In [5]:
# ======================================
# 2. Data Directories
# ======================================
train_data_config = {
    "turnaround": {
        "images": str(PROJECT_ROOT / "data/processed/train/turnaround/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/turnaround/labels")
    },
    "ppe": {
        "images": str(PROJECT_ROOT / "data/processed/train/ppe/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/ppe/labels")
    },
    "fod": {
        "images": str(PROJECT_ROOT / "data/processed/train/fod/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/fod/labels")
    }
}

val_data_config = {
    "turnaround": {
        "images": str(PROJECT_ROOT / "data/processed/val/turnaround/images"),
        "labels": str(PROJECT_ROOT / "data/processed/val/turnaround/labels")
    },
    "ppe": {
        "images": str(PROJECT_ROOT / "data/processed/val/ppe/images"),
        "labels": str(PROJECT_ROOT / "data/processed/val/ppe/labels")
    },
    "fod": {
        "images": str(PROJECT_ROOT / "data/processed/val/fod/images"),
        "labels": str(PROJECT_ROOT / "data/processed/val/fod/labels")
    }
}


In [6]:
# ======================================
# 3. Create Multi-Task DataLoaders
# ======================================
print("\n--- 📦 Initializing DataLoaders ---")
train_loader = create_multitask_dataloader(
    data_dirs=train_data_config,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader = create_multitask_dataloader(
    data_dirs=val_data_config,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)


--- 📦 Initializing DataLoaders ---


📦 Loaded [TURNAROUND] dataset: 6813 samples
📦 Loaded [PPE] dataset: 5641 samples
📦 Loaded [FOD] dataset: 23655 samples
✅ Combined Multi-Task Dataset total size: 36109 samples

📦 Loaded [TURNAROUND] dataset: 1460 samples
📦 Loaded [PPE] dataset: 1209 samples
📦 Loaded [FOD] dataset: 5069 samples
✅ Combined Multi-Task Dataset total size: 7738 samples



In [7]:
# ======================================
# 4. Initialize Architecture 
# & Auto-Balancing Loss
# ======================================
print("\n--- 🏗️ Initializing Model Architecture & Loss Engine ---")
config = MultiTaskModelConfig()
model = MultiTaskYOLO(config)
loss_fn = MultiTaskUncertaintyLoss(model, config)

# ======================================
# 5. Optimizer & Cosine Annealing
# Learning Rate Scheduler
# ======================================
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

# ======================================
# 6. Instantiaye Trainer Engine
# ======================================
trainer = MultiTaskTrainer(
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    device=DEVICE,
    use_wandb=False,    # Set to True if WandB is logged in
    exp_name="full_multitask_30ep_run",
    checkpoint_dir="checkpoints"
)

# ======================================
# 7. Luanch Training Schedule
# ======================================
print("\n--- 🚀 Launching Full Multi-Task Training ---")
trainer.fit(num_epochs=NUM_EPOCHS)



--- 🏗️ Initializing Model Architecture & Loss Engine ---



--- 🚀 Launching Full Multi-Task Training ---
🚀 Starting Multi-Task Model Training on device [CUDA] for 50 Epcohs...



Epoch [01/50] | Train Loss: 439.3497 | Val Loss: 52.6601
 💾 Saved Best Model Checkpoint (Val Loss: 52.6601) -> /teamspace/studios/this_studio/aiport-incident-detection-final/model/checkpoints/best_multitask_model.pt

✅ Multi-Task Training Pipeline Completed Successfully!


Epoch [02/50] | Train Loss: 43.3516 | Val Loss: 39.8563
 💾 Saved Best Model Checkpoint (Val Loss: 39.8563) -> /teamspace/studios/this_studio/aiport-incident-detection-final/model/checkpoints/best_multitask_model.pt

✅ Multi-Task Training Pipeline Completed Successfully!


Epoch [03/50] | Train Loss: 29.7125 | Val Loss: 25.9965
 💾 Saved Best Model Checkpoint (Val Loss: 25.9965) -> /teamspace/studios/this_studio/aiport-incident-detection-final/model/checkpoints/best_multitask_model.pt

✅ Multi-Task Training Pipeline Completed Successfully!


...

Epoch [31/50] | Train Loss: 5.9442 | Val Loss: 6.5382

✅ Multi-Task Training Pipeline Completed Successfully!


Epoch [32/50] | Train Loss: 5.9382 | Val Loss: 6.5235

✅ Multi-Task Training Pipeline Completed Successfully!


KeyboardInterrupt: 


--- 🏗️ Initializing Model Architecture & Loss Engine ---

--- 🚀 Launching Full Multi-Task Training ---
🚀 Starting Multi-Task Model Training on device [CUDA] for 50 Epcohs...



KeyboardInterrupt: 

In [ ]:
# ==============================
# Evaluation on Test Dataset
# ==============================
test_data_config = {
    "turnaround": {
        "images": str(PROJECT_ROOT / "data/processed/test/turnaround/images"),
        "labels": str(PROJECT_ROOT / "data/processed/val/turnaround/labels")
    },
    "ppe": {
        "images": str(PROJECT_ROOT / "data/processed/test/ppe/images"),
        "labels": str(PROJECT_ROOT / "data/processed/test/ppe/labels")
    },
    "fod": {
        "images": str(PROJECT_ROOT / "data/processed/test/fod/images"),
        "labels": str(PROJECT_ROOT / "data/processed/test/fod/labels")
    }
}

checkpoint_file = str(PROJECT_ROOT / "model/checkpoints/best_multitask_model.pt")

# Run metric evaluation
metrics = run_evaluation(checkpoint_path=checkpoint_file, test_data_config=test_data_config)

In [ ]:
# ====================================
# Run Multi-Stream Video Inference
# ====================================

# Define video stream inputs (or sample video files/RTSP streams)
stream_sources = {
    "turnaround": "data/samples/turnaround_sample.mp4",
    "ppe": "data/samples/ppe_sample.mp4",
    "fod": "data/samples/fod_sample.mp4"
}

# Instantiate Engine
processor = AirportStreamProcessor(
    checkpoint_path="checkpoints/best_multitask_model.pt",
    conf_thresh=0.30
)

# Run Inference across 3 Streams and output tiles view
processor.process_triple_stream(
    stream_sources=stream_sources,
    output_video_path="output/airport_incident_multistream.mp4",
    max_frames=200
)